# Qwen2.5-VL-3B LoRA 微调（考古断代）
在 **Google Colab 免费 GPU** 上运行。需先把 `colab_bundle.zip`（含 `train.jsonl` + `images/`，由 `lora_data.py` + `lora_package.py` 生成）上传到本 Colab 会话（左侧文件/下方 `files.upload()`）。

运行后产物：`/content/lora_out/adapter`（PEFT adapter）+ `/content/lora_out/gguf`（GGUF 文件，供本地 Ollama 部署）。

> 基座固定为 `Qwen/Qwen2.5-VL-3B-Instruct`，与本机 Ollama 的 `qwen2.5-vl:3b` 一致，便于落地。

In [ ]:
# 1) 安装依赖（Colab 每会话需重装）
!pip install -q "torch>=2.3" "transformers>=4.49" "peft" "accelerate" "bitsandbytes" "datasets" "pillow" "unsloth"

In [ ]:
# 2) 上传并解压数据包（或在左侧手动上传后跳过 extract，直接指向目录）
import os, zipfile
if not os.path.exists("/content/colab_bundle"):
    os.makedirs("/content/colab_bundle", exist_ok=True)
    if os.path.exists("/content/colab_bundle.zip"):
        with zipfile.ZipFile("/content/colab_bundle.zip") as z:
            z.extractall("/content")
        print("已解压 colab_bundle")
DATA = "/content/colab_bundle/train.jsonl"
print("数据路径:", DATA)

In [ ]:
# 3) 数据集处理：LLaVA/Qwen2-VL conversations -> processor chat 格式，仅监督 assistant
import json, torch
from PIL import Image
from datasets import Dataset
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoProcessor

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

BASE = "Qwen/Qwen2.5-VL-3B-Instruct"
processor = AutoProcessor.from_pretrained(BASE, trust_remote_code=True)
samples = load_jsonl(DATA)
print("样本数:", len(samples))

def process_one(s):
    img = Image.open(s["image"]).convert("RGB")
    user = s["conversations"][0]["value"].replace("<image>", "").strip()
    ans = s["conversations"][1]["value"]
    messages = [
        {"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": user}]},
        {"role": "assistant", "content": [{"type": "text", "text": ans}]},
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    inputs = processor(text=[text], images=[img], return_tensors="pt", padding=True, max_length=2048)
    user_text = processor.apply_chat_template(
        [{"role": "user", "content": [{"type": "text", "text": user}]}], tokenize=False, add_generation_prompt=False)
    user_len = len(processor.tokenizer(user_text, add_special_tokens=False)["input_ids"])
    input_ids = inputs["input_ids"][0]
    labels = input_ids.clone()
    labels[:user_len] = -100
    return {"input_ids": input_ids, "attention_mask": inputs["attention_mask"][0], "labels": labels,
            "pixel_values": inputs["pixel_values"][0], "image_grid_thw": inputs["image_grid_thw"][0]}

def collate(batch):
    pad = lambda seq, pid: pad_sequence(seq, batch_first=True, padding_value=pid)
    input_ids = pad([b["input_ids"] for b in batch], processor.tokenizer.pad_token_id)
    attention_mask = pad([b["attention_mask"] for b in batch], 0)
    labels = pad([b["labels"] for b in batch], -100)
    maxp = max(b["pixel_values"].shape[1] for b in batch)
    pixel_values = torch.stack([torch.nn.functional.pad(b["pixel_values"], (0,0,0,maxp-b["pixel_values"].shape[1]), value=0) for b in batch])
    image_grid_thw = torch.cat([b["image_grid_thw"] for b in batch], dim=0)
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels,
            "pixel_values": pixel_values, "image_grid_thw": image_grid_thw}

ds = Dataset.from_list(samples)
ds = ds.map(lambda x: process_one(x), remove_columns=ds.column_names)
print("数据集已处理")

In [ ]:
# 4) 加载基座 + QLoRA（4bit）
import torch
from transformers import AutoModelForVision2Seq, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForVision2Seq.from_pretrained(BASE, trust_remote_code=True, quantization_config=bnb)
lora = LoraConfig(r=16, lora_alpha=16, lora_dropout=0.0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"], bias="none", task_type="CAUSAL_LM")
model = get_peft_model(model, lora)
model.config.use_cache = False
model.print_trainable_parameters()

In [ ]:
# 5) 训练
from transformers import Trainer, TrainingArguments
args = TrainingArguments(
    output_dir="/content/lora_out", num_train_epochs=3, learning_rate=2e-4,
    per_device_train_batch_size=1, gradient_accumulation_steps=8,
    logging_steps=10, save_strategy="epoch", report_to="none",
    bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(),
    remove_unused_columns=False)
trainer = Trainer(model=model, args=args, train_dataset=ds, data_collator=collate)
trainer.train()
model.save_pretrained("/content/lora_out/adapter")
processor.save_pretrained("/content/lora_out/adapter")
print("adapter 已保存到 /content/lora_out/adapter")

In [ ]:
# 6) 导出 GGUF 供本地 Ollama 部署（用 unsloth 合并导出；或下载 adapter 手动转 GGUF）
from unsloth import FastVisionModel
model, tokenizer = FastVisionModel.from_pretrained(BASE, load_in_4bit=True)
model.load_adapter("/content/lora_out/adapter")
model.save_pretrained_gguf("/content/lora_out/gguf", quant_method="q4_k_m")
print("GGUF 已导出到 /content/lora_out/gguf")

In [ ]:
# 7) 下载产物
import shutil, os
shutil.make_archive("/content/lora_adapter", "zip", "/content/lora_out/adapter")
shutil.make_archive("/content/lora_gguf", "zip", "/content/lora_out/gguf")
from google.colab import files
files.download("/content/lora_adapter.zip")
files.download("/content/lora_gguf.zip")

## 本地部署
1. 把 `/content/lora_gguf.zip` 解压出的 GGUF 拷到本机 `ollama-models/lora_model.gguf`。
2. 在本机已有基座 `qwen2.5-vl:3b` 的前提下，用 `deploy/ollama/qwen2.5-vl-lora.Modelfile` 创建微调模型：
   `ollama create qwen2.5-vl-lora -f deploy/ollama/qwen2.5-vl-lora.Modelfile`
3. 传图测试：`ollama run qwen2.5-vl-lora`。
   若无法用 `ADAPTER`，可在 Colab 把 LoRA merge 进基座并导出含 mmproj 的完整 GGUF，按 5.1「双 FROM」方式创建。